<a href="https://colab.research.google.com/github/unknownexplosion/Sentiment-analysis/blob/main/ABSA_Fine_Tuning_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tune DeBERTa for Aspect-Based Sentiment Analysis

This notebook trains a **true aspect-conditioned** DeBERTa v3 classifier from scratch.

**How it works:**
- Input is a sentence-pair: `[CLS] clause_text [SEP] aspect_name [SEP]`
- The model learns to classify sentiment *for a specific aspect* in the text
- So the same sentence can get different labels for different aspects

**Requirements:**
- Google Colab with **GPU runtime** (T4 is fine)
- Upload `absa_training_dataset.csv` when prompted
- Your Hugging Face **write token** for model upload

In [ ]:
# 1. Install Dependencies
!pip install -q transformers torch scikit-learn huggingface_hub accelerate sentencepiece

In [ ]:
# 2. Upload your training dataset
# Run this cell, then click "Choose Files" and select:
#   outputs/absa_training_dataset.csv
from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_csv('absa_training_dataset.csv')
print(f'\nLoaded {len(df)} rows')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nAspect distribution:')
print(df['aspect'].value_counts())

In [ ]:
# 3. Train the Aspect-Based Sentiment Classifier

import os
import json
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# ── Config ──
BASE_MODEL = "microsoft/deberta-v3-small"  # Fresh base model
OUTPUT_DIR = "fine_tuned_absa_model"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 8

# ── Prepare data ──
df = df[df['label'].isin(['Positive', 'Negative', 'Neutral'])]
df = df.drop_duplicates(subset=['text', 'aspect'])
print(f'Training samples after dedup: {len(df)}')

label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df['label_id'] = df['label'].map(label_map)

# Class weights for imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=df['label_id'].values
)
print(f'Class weights: Neg={class_weights[0]:.3f}, Neu={class_weights[1]:.3f}, Pos={class_weights[2]:.3f}')

# Train/val split
train_texts, val_texts, train_asp, val_asp, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['aspect'].tolist(),
    df['label_id'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label_id'].tolist(),
)
print(f'Train: {len(train_texts)}, Val: {len(val_texts)}')

# ── Tokenize (sentence pair: text + aspect) ──
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
train_enc = tokenizer(train_texts, train_asp, truncation=True, padding=True, max_length=MAX_LEN)
val_enc   = tokenizer(val_texts,   val_asp,   truncation=True, padding=True, max_length=MAX_LEN)

# ── Dataset class ──
class ABSADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ABSADataset(train_enc, train_labels)
val_dataset   = ABSADataset(val_enc,   val_labels)

# ── Weighted Trainer ──
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            weights = torch.tensor(self.class_weights, dtype=torch.float, device=logits.device)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fn = torch.nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Metrics ──
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# ── Model ──
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label={0: 'Negative', 1: 'Neutral', 2: 'Positive'},
    label2id=label_map,
)

# ── Training Args ──
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    fp16=True,  # Enable mixed precision on GPU
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

# ── Train! ──
print('\n🚀 Starting training...')
trainer.train()

# ── Evaluate ──
eval_results = trainer.evaluate()
print(f'\n📊 Final Evaluation:')
print(f'  Accuracy:  {eval_results["eval_accuracy"]:.4f}')
print(f'  F1-Score:  {eval_results["eval_f1"]:.4f}')
print(f'  Precision: {eval_results["eval_precision"]:.4f}')

# Save model + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save metrics
with open(f'{OUTPUT_DIR}/metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f'\n✅ Model saved to {OUTPUT_DIR}/')

In [ ]:
# 4. Validate Aspect-Conditioned Inference
# This is the KEY test — the model should produce DIFFERENT sentiments
# for different aspects on the SAME contrastive sentence.

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model.eval()

id2label = model.config.id2label

def predict(text, aspect):
    inputs = tokenizer(text, aspect, truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = probs.argmax().item()
    return id2label[pred_id], probs[pred_id].item()

# Test contrastive sentences
tests = [
    ('The battery is amazing', 'Battery', 'Positive'),
    ('the screen scratches easily', 'Display', 'Negative'),
    ('Camera quality is excellent', 'Camera', 'Positive'),
    ('the price is way too high', 'Price', 'Negative'),
    ('Performance is buttery smooth', 'Performance', 'Positive'),
    ('the device overheats during gaming', 'Heating / Thermals', 'Negative'),
    ('The battery is terrible', 'Battery', 'Negative'),
    ('The display is gorgeous', 'Display', 'Positive'),
]

print('Aspect-Conditioned Inference Validation')
print('=' * 70)
correct = 0
for text, aspect, expected in tests:
    label, conf = predict(text, aspect)
    match = '✅' if label == expected else '❌'
    if label == expected:
        correct += 1
    print(f'{match} [{aspect:>20}] "{text}" → {label} ({conf:.4f}) (expected: {expected})')

print(f'\nAccuracy: {correct}/{len(tests)} ({correct/len(tests)*100:.0f}%)')

In [ ]:
# 5. Upload to Hugging Face Hub

from huggingface_hub import HfApi, login

# Enter your Hugging Face write token
HF_TOKEN = input('Enter your Hugging Face write token: ').strip()
REPO_ID  = 'unknownexplosion/SentimentAnalysisog'  # Your existing repo

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type='model',
)

print(f'\n🚀 Model uploaded to https://huggingface.co/{REPO_ID}')
print(f'\nYour Streamlit app will now use the new aspect-conditioned model!')

In [ ]:
# 6. (Alternative) Download the model as a zip file
# Use this if you prefer to upload manually later
!zip -r fine_tuned_absa_model.zip fine_tuned_absa_model/

from google.colab import files
files.download('fine_tuned_absa_model.zip')